# Day 95 — CNN Architecture

This notebook covers Modular Convolutional Neural Network Architecture (`SimpleCNN`), Two-Stage Feature Extractor & Classifier Head Design, Dynamic Flattened Dimension Calculation, Raw Logits Output & `nn.CrossEntropyLoss` Numerical Stability, and Channel-vs-Spatial Scaling Trade-Offs

## Validation goals
- Structure a two-stage CNN separating stacked convolutional feature extraction (`Conv2d -> ReLU -> MaxPool2d`) from a flattened fully connected 
- Implement `SimpleCNN` with automatic spatial tracking (`spatial = spatial // 2`) and guardrails to compute `flattened_dim = prev_c * spatial * spatial` dynamically across arbitrary input resolutions and channel widths
- Trace batch tensor transformations from `(8, 3, 32, 32)` through `model.features` (`(8, 128, 4, 4)`) to raw class logits (`(8, 10)`) and audit total trainable parameter counts (`357,066` parameters)
- Analyze why channel depth expands as spatial height/width shrinks, how flattened dimensions map into `nn.Linear`, and why raw logits outperform an explicit `Softmax` layer when paired with `nn.CrossEntropyLoss`

### 1. Two-Stage Structure of a Standard CNN

A classic Convolutional Neural Network splits into two clear stages:
- Feature Extractor (Convolutional Backbone): Stacks [`Conv2d -> BatchNorm2d (optional) -> ReLU -> MaxPool2d`] blocks. Early blocks detect low-level edges and color contrast at high spatial resolution; deeper blocks expand channel width (C increases) while shrinking spatial dimensions (H, W decrease) to capture higher-level textures and parts.
- Task Head (Flatten + Fully Connected Layers): Converts 3D spatial feature maps (C, H, W) into a 1D feature vector via `.flatten(start_dim=1)` (or `AdaptiveAvgPool2d((1, 1))`), then maps through nn.Linear layers to produce class logits (num_classes outputs) or a continuous value (1 output).

### 2. Tracing Shapes Through a CNN Block

Tracking (N, C, H, W) at every boundary prevents dimension mismatches at the Linear layer. Consider a 32x32 RGB input (N, 3, 32, 32):
- Block 1: `Conv2d(3 -> 32, kernel_size=3, padding=1) + ReLU + MaxPool2d(2, 2) -> (N, 32, 16, 16)`
- Block 2: `Conv2d(32 -> 64, kernel_size=3, padding=1) + ReLU + MaxPool2d(2, 2) -> (N, 64, 8, 8)`
- Block 3: `Conv2d(64 -> 128, kernel_size=3, padding=1) + ReLU + MaxPool2d(2, 2) -> (N, 128, 4, 4)`
- Flatten & Head: `128 * 4 * 4 = 2048 features -> Linear(2048 -> 128) + ReLU -> Linear(128 -> num_classes)`

### 3. Implementing a Modular CNN

Let's write a reusable `SimpleCNN` class that exposes `in_features` calculation cleanly so any image size (`32x32`, `64x64`, etc.) or channel count (`1 grayscale, 3 RGB`) wires into the classifier head without hardcoding magic numbers.

In [ ]:
import torch
import torch.nn as nn


class SimpleCNN(nn.Module):
    """Modular CNN for image classification with configurable channels and input size."""

    def __init__(
        self,
        in_channels: int = 3,
        num_classes: int = 10,
        img_size: int = 32,
        channel_dims: tuple[int, ...] = (32, 64, 128),
        hidden_fc: int = 128,
        dropout: float = 0.2,
    ):
        super().__init__()
        if not channel_dims:
            raise ValueError("channel_dims must contain at least one channel width.")

        blocks: list[nn.Module] = []
        prev_c = in_channels
        spatial = img_size

        for out_c in channel_dims:
            blocks.append(nn.Conv2d(prev_c, out_c, kernel_size=3, stride=1, padding=1))
            blocks.append(nn.ReLU())
            blocks.append(nn.MaxPool2d(kernel_size=2, stride=2))
            spatial = spatial // 2
            prev_c = out_c

        if spatial < 1:
            raise ValueError("Input image size is too small for the number of pooling layers.")

        self.features = nn.Sequential(*blocks)
        flattened_dim = prev_c * spatial * spatial

        self.classifier = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(flattened_dim, hidden_fc),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_fc, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


if __name__ == "__main__":
    torch.manual_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = SimpleCNN(
        in_channels=3,
        num_classes=10,
        img_size=32,
        channel_dims=(32, 64, 128),
        hidden_fc=128,
        dropout=0.2,
    ).to(device)

    batch_x = torch.randn((8, 3, 32, 32), dtype=torch.float32, device=device)

    with torch.no_grad():
        feat_map = model.features(batch_x)
        logits = model(batch_x)

    print("--- Day 95: SimpleCNN Architecture ---")
    print(f"Device:             {device}")
    print(f"Input batch shape:  {tuple(batch_x.shape)}")
    print(f"Feature map shape:  {tuple(feat_map.shape)}")
    print(f"Logits output shape:{tuple(logits.shape)}")
    print(f"Trainable params:   {count_params(model):,}")

### 4. Architectural Design Notes

Raw Logits at the Head: For multi-class classification, `SimpleCNN` returns raw logits of shape `(N, num_classes)` rather than applying Softmax inside f`orward()`, because `nn.CrossEntropyLoss()` combines `LogSoftmax` and `NLLLoss` in one numerically stable step.

Why Channels Grow While Spatial Size Shrinks: Halving height and width cuts spatial positions by $4\times$, so expanding channels (`32 -> 64 -> 128`) preserves rich semantic capacity without letting late Linear layers blow up in parameter count.Where Most Parameters Live: Notice that `nn.Linear(2048 -> 128)` alone holds `262,272 `weights—often more than all the convolutional layers combined. Using AdaptiveAvgPool2d((1, 1)) before classification is a common alternative when you want fewer head parameters.

### Guided Question:

In a `CNN`, why do we typically increase out_channels across deeper blocks while reducing spatial height and width via pooling or striding?

### Answer:

Increasing `out_channels` while shrinking spatial height ($H$) and width ($W$) in deeper Convolutional Neural Network (CNN) blocks balances semantic richness against compute and memory cost. As spatial resolution drops, each pixel represents a larger region of the original image, so more feature channels are needed to describe what is inside that broader view.

**1. Receptive Field Growth vs. Semantic Diversity**

- Early layers (small receptive field, fewer channels): A $3 \times 3$ kernel in an early block looks at a tiny patch of pixels. The patterns available at this scale—edges, corners, color transitions, and basic textures—are relatively few, so a modest channel count (such as 32 or 64) covers them well.
- Deeper layers (large receptive field, more channels): After pooling or strided convolutions, a single spatial position in a deep feature map corresponds to a wide region of the input image. That region can contain complex combinations of parts—such as an eye, a wheel, a doorway, or a specific object motif—in many poses and contexts. More channels give the network capacity to represent distinct high-level concepts and part combinations simultaneously.

**2. Trading Space for Channel Capacity Within Compute Budgets**

A standard convolutional layer's FLOPs scale linearly with spatial area ($H \times W$), input channels ($C_{\text{in}}$), output channels ($C_{\text{out}}$), and kernel area ($K_h \times K_w$):$$\text{FLOPs} \propto H \times W \times C_{\text{in}} \times C_{\text{out}} \times K_h \times K_w$$
- Halving both $H$ and $W$ reduces the number of spatial positions by a factor of $4$ ($2 \times 2$).
- That spatial reduction creates headroom to increase $C_{\text{out}}$ (often doubling it) without blowing up activation memory or runtime.
- Keeping both high spatial resolution and high channel counts in early stages is expensive because early feature maps have the largest $H \times W$ dimensions; keeping early channels low keeps early activations lightweight.

**3. Spatial Invariance and Noise Reduction**

- Local translation tolerance: Max pooling or strided convolution discards exact sub-patch coordinates within each downsampled window, helping downstream layers respond consistently even if a feature shifts slightly.
- Redundancy across features: Having multiple channels at coarser spatial scales lets different filters specialize in variations of a concept (for example, different orientations, lighting conditions, or sub-parts) rather than forcing a single channel to carry all variations at once.

**4. Bottlenecks and Practical Trade-Offs**

- Information loss from aggressive downsampling: Shrinking $H$ and $W$ too quickly can erase fine spatial details needed for tasks like semantic segmentation, small-object detection, or precise localization.
- Parameter growth in deep layers: Because each filter in layer $l$ has depth $C_{\text{in}}$, doubling channels in every block increases weight count quickly. Modern architectures often pair channel expansion with $1 \times 1$ convolutions (bottlenecks) or depthwise separable convolutions to control parameters and FLOPs while preserving representational breadth

### Guided Question:

If `model.features(x)` outputs shape `(16, 64, 8, 8)`, what input width does the first nn.Linear layer need after `nn.Flatten(start_dim=1)`?

### Answer:

The first `nn.Linear` layer needs an input width `(in_features)` of `4096.Shape `

**Calculation**

- Input from `model.features(x)`: `(16, 64, 8, 8) `representing (batch_size, channels, height, width).
- After `nn.Flatten(start_dim=1)`: Dimension 0 `(batch size 16)` stays intact, and dimensions 1 through 3 multiply together:$$64 \times 8 \times 8 = 4096$$
- Flattened tensor shape: `(16, 4096)`.

### Guided Question:

Why does a PyTorch classification network typically output raw logits from its final `nn.Linear` layer instead of a `Softmax` layer?

### Answer:

Outputting raw logits (unnormalized scores) from the final `nn.Linear` layer and applying `Softmax` inside the loss function is cleaner, faster, and numerically safer in PyTorch.

**1. Numerical Stability in Loss Computation**

`nn.CrossEntropyLoss` combines `LogSoftmax` and `NLLLoss` into a single fused operation using the log-sum-exp trick (subtracting the maximum logit before exponentiating):$$\text{LogSoftmax}(z_i) = z_i - \log \sum_j \exp(z_j)$$
- If a model outputs large positive or negative logits, c`omputing exp(z_j)` directly in a separate Softmax layer can overflow to inf or underflow to 0.0, producing NaN gradients during backpropagation.
- Fusing `LogSoftmax` and NLLLoss cancels terms algebraically and handles extreme values reliably in both float32 and mixed precision `(torch.amp / float16 or bfloat16)`.

**2. Avoiding Double Softmax Bugs**

`nn.CrossEntropyLoss` expects raw logits as input.

- If you append `nn.Softmax(dim=1)` to the model and pass the output to nn.`CrossEntropyLoss`, the network applies Softmax twice—distorting probabilities, flattening gradients, and degrading training.
- Keeping the model's forward pass focused on logits avoids this mismatch.

**3. Faster Execution and Cleaner Inference**

- Fewer kernels during training: Fusing LogSoftmax and NLLLoss avoids extra intermediate tensor allocations and extra pass-throughs over memory.
- Flexible inference: During evaluation or deployment, you can call torch.softmax`(logits, dim=-1)` (or `torch.argmax(logits, dim=-1)` when you only need the predicted class index, since Softmax is strictly order-preserving on class scores).